# RSNA Knee — Step 3: Label Extraction Pilot

**Settings: Accelerator = `GPU T4 x2`, Internet = `On`.** Attach the competition dataset.

Only 58 of 4,407 studies carry expert labels. Every study has a report. So the reports *are* the labels — for everyone in this competition.

This notebook answers the question the whole season rests on: **how accurately can the twelve findings be recovered from report text?**

We run an open-weights multilingual LLM over the 58 gold reports and score its output against the expert labels. Running it locally on Kaggle matters — the reports never leave the platform, so the data-redistribution clause in the rules (Section 2.4.b) never comes into play.

Runtime ~10 min, mostly weight download.

In [ ]:
import os, re, json, time, warnings, gc
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import torch

ROOT   = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
train  = pd.read_csv(f"{ROOT}/train.csv")
LABELS = pd.read_csv(f"{ROOT}/sample_submission.csv").columns[1:].tolist()

gold = train[train[LABELS].notna().sum(axis=1) == len(LABELS)].reset_index(drop=True)
print(f"gold studies: {len(gold)}  |  labels: {len(LABELS)}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set Accelerator!'}")

## The prompt

Three things it has to get right, all of which came straight out of the EDA:

- **Negation.** `no` appears in 43% of reports, `normal` in 47%, `intact` in 24%. The anatomy is described whether or not it's damaged, so polarity carries all the signal.
- **Multilingual.** Nine languages, and only ~20% of reports have a recognisable English section header. We read in the original language rather than translating.
- **Medial vs lateral.** Four labels depend on getting the compartment right.

We also ask for OA **severity**, because `mild`/`moderate`/`severe` appear in 19%/9%/5% of reports — meaning osteoarthritis is graded, and somebody chose a threshold when making the gold labels. We need to recover that threshold.

In [ ]:
KEYMAP = {
    "acl":"ACL", "mcl":"MCL",
    "medial_meniscus":"Medial Meniscus", "lateral_meniscus":"Lateral Meniscus",
    "medial_oa":"Medial OA", "lateral_oa":"Lateral OA", "pf_oa":"PF OA",
    "effusion":"Effusion", "synovitis":"Synovitis", "bakers":"Baker's",
    "contusion":"Contusion", "fracture":"Fracture",
}

SYSTEM = """You are a musculoskeletal radiologist extracting structured findings from knee MRI reports.
The report may be written in ANY language (English, Spanish, Portuguese, German, French, Italian, Turkish, Dutch, and others). Read it in its original language.

For each of the twelve findings below, decide whether the report states it is PRESENT, states it is ABSENT/normal/intact, or does NOT MENTION it at all.

Findings:
- acl: anterior cruciate ligament tear or injury
- mcl: medial collateral ligament tear or injury
- medial_meniscus: medial meniscus tear
- lateral_meniscus: lateral meniscus tear
- medial_oa: osteoarthritis / chondral loss of the MEDIAL tibiofemoral compartment
- lateral_oa: osteoarthritis / chondral loss of the LATERAL tibiofemoral compartment
- pf_oa: patellofemoral osteoarthritis / chondral loss
- effusion: joint effusion, excess joint fluid
- synovitis: synovial thickening or inflammation
- bakers: Baker's cyst / popliteal cyst
- contusion: bone contusion, bone bruise, bone marrow oedema from trauma
- fracture: fracture of any bone

CRITICAL RULES:
1. Negation matters. "ACL is intact", "menisci are normal", "no tear" => "absent".
2. Degenerative/mucoid change or signal that is NOT a tear => "absent" for meniscus/ligament.
3. Distinguish MEDIAL from LATERAL compartments carefully. Only mark a compartment if that specific side is named, or if a global statement clearly covers it.
4. If the finding is never referred to, use "not_mentioned". Do NOT guess.
5. For the three osteoarthritis findings also give severity: "none", "mild", "moderate", or "severe".

Reply with ONLY a JSON object, no prose:
{"acl":"present|absent|not_mentioned", "mcl":"...", "medial_meniscus":"...", "lateral_meniscus":"...", "medial_oa":"...", "lateral_oa":"...", "pf_oa":"...", "effusion":"...", "synovitis":"...", "bakers":"...", "contusion":"...", "fracture":"...", "medial_oa_severity":"none|mild|moderate|severe", "lateral_oa_severity":"...", "pf_oa_severity":"..."}"""

print("prompt ready")

## Load the model

Qwen2.5-7B-Instruct: open weights, strong multilingual, free, and permitted under Section 2.6 (publicly available pretrained model, equally accessible to all participants).

Loaded in 4-bit so it fits comfortably and leaves headroom. Greedy decoding — we want reproducible extractions, not creative ones.

In [ ]:
MODEL = "Qwen/Qwen2.5-7B-Instruct"
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f"loading {MODEL} ...")
t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16)
model.eval()
print(f"loaded in {time.time()-t0:.0f}s")


def extract(report, max_new_tokens=300):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user",   "content": f"REPORT:\n{report[:6000]}"}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids  = tok([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                             temperature=None, top_p=None, top_k=None,
                             pad_token_id=tok.eos_token_id)
    gen = tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
    m = re.search(r"\{.*\}", gen, re.S)
    if not m:
        return None, gen
    try:
        return json.loads(m.group(0)), gen
    except Exception:
        return None, gen

print("ready")

## Run over the 58 gold reports

In [ ]:
preds, raws, fails = [], [], 0
t0 = time.time()
for i, r in gold.iterrows():
    p, raw = extract(str(r["Report"]))
    if p is None:
        fails += 1
    preds.append(p or {})
    raws.append(raw)
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(gold)}   ({(time.time()-t0)/(i+1):.1f}s each)")

el = time.time() - t0
print(f"\ndone in {el/60:.1f} min | {el/len(gold):.1f}s per report | JSON parse failures: {fails}")

## Score against gold

`precision` and `recall` are reported separately because they fail differently. Low **recall** means the extractor missed a finding the report mentions — usually fixable with prompt work. Low **precision** means it invented findings, which is far more damaging: it poisons the training labels for 4,349 studies.

In [ ]:
def to_bin(v, unmentioned=0):
    if v == "present": return 1
    if v == "absent":  return 0
    return unmentioned

print("="*80)
print(f"PER-LABEL AGREEMENT vs GOLD   (n={len(gold)})")
print("="*80)
print(f"{'label':<18}{'acc':>7}{'TP':>5}{'FP':>5}{'FN':>5}{'TN':>5}{'prec':>7}{'rec':>7}{'unment':>8}")

rows = []
for key, col in KEYMAP.items():
    y     = pd.to_numeric(gold[col], errors="coerce").fillna(0).astype(int).values
    raw_v = [p.get(key, "not_mentioned") for p in preds]
    yh    = np.array([to_bin(v) for v in raw_v])
    unm   = sum(1 for v in raw_v if v not in ("present", "absent"))
    tp = int(((y==1)&(yh==1)).sum()); fp = int(((y==0)&(yh==1)).sum())
    fn = int(((y==1)&(yh==0)).sum()); tn = int(((y==0)&(yh==0)).sum())
    acc  = (tp+tn)/len(y)
    prec = tp/(tp+fp) if tp+fp else float("nan")
    rec  = tp/(tp+fn) if tp+fn else float("nan")
    rows.append(dict(label=col, acc=acc, prec=prec, rec=rec, tp=tp, fp=fp, fn=fn, tn=tn, unm=unm))
    print(f"{col:<18}{acc:>7.2f}{tp:>5}{fp:>5}{fn:>5}{tn:>5}{prec:>7.2f}{rec:>7.2f}{unm:>8}")

res = pd.DataFrame(rows)
print(f"\nmean accuracy across 12 labels : {res.acc.mean():.3f}")
print(f"labels below 0.80 accuracy     : {list(res[res.acc < 0.80].label)}")

## Does "not mentioned" mean absent?

Radiologists don't enumerate every negative. Silence about synovitis might mean it's absent, or might mean they never assessed it. The right interpretation probably differs by finding — fracture is usually called out explicitly, synovitis often isn't.

This scores both readings so you can choose per label rather than guessing globally.

In [ ]:
print("--- not_mentioned -> 0  vs  not_mentioned -> 1 ---")
for key, col in KEYMAP.items():
    y = pd.to_numeric(gold[col], errors="coerce").fillna(0).astype(int).values
    a = np.array([to_bin(p.get(key, "not_mentioned"), 0) for p in preds])
    b = np.array([to_bin(p.get(key, "not_mentioned"), 1) for p in preds])
    if (a != b).sum():
        print(f"  {col:<18} ->0: {(y==a).mean():.2f}   ->1: {(y==b).mean():.2f}   (n_differ={int((a!=b).sum())})")

## Recovering the OA threshold

`mild` appears in 19% of reports, `moderate` 9%, `severe` 5% — so osteoarthritis is graded, not binary. Whoever produced the gold labels drew a line somewhere.

If `mild` shows a gold positive rate near 0.2 and `moderate` near 0.9, the threshold sits between them, and you've just fixed three of your twelve labels.

In [ ]:
for key, col in [("medial_oa","Medial OA"), ("lateral_oa","Lateral OA"), ("pf_oa","PF OA")]:
    y   = pd.to_numeric(gold[col], errors="coerce").fillna(0).astype(int).values
    sev = [str(p.get(f"{key}_severity", "none")).lower() for p in preds]
    print(f"\n{col}:")
    for s in ["none", "mild", "moderate", "severe"]:
        idx = [i for i, v in enumerate(sev) if v == s]
        if idx:
            print(f"   severity={s:<9} n={len(idx):>3}   gold positive rate = {y[idx].mean():.2f}")

In [ ]:
pd.DataFrame(preds).assign(StudyInstanceUID=gold.StudyInstanceUID).to_csv("gold58_extraction.csv", index=False)
res.to_csv("extraction_scores.csv", index=False)
print("wrote gold58_extraction.csv + extraction_scores.csv")

## PatientID leakage check

`PatientID` survived de-identification. If one patient contributed several studies — both knees, or a follow-up — and they land in different CV folds, your validation score is inflated by leakage and you won't find out until the private leaderboard.

If this reports any patient with more than one study, group your folds by `PatientID` rather than `StudyInstanceUID`.

In [ ]:
import pydicom, glob, collections

sdirs = sorted(glob.glob(f"{ROOT}/train_series/*/*"))
rng   = np.random.default_rng(0)
pick  = rng.choice(len(sdirs), size=min(600, len(sdirs)), replace=False)

pid_of = {}
for i in pick:
    d  = sdirs[i]
    fs = os.listdir(d)
    if not fs: continue
    try:
        ds = pydicom.dcmread(os.path.join(d, fs[0]), stop_before_pixels=True)
        pid_of[str(ds.StudyInstanceUID)] = str(ds.PatientID)
    except Exception:
        pass

n_std, n_pid = len(pid_of), len(set(pid_of.values()))
print(f"sampled studies: {n_std}  |  distinct PatientIDs: {n_pid}")
if n_pid < n_std:
    c = collections.Counter(pid_of.values())
    multi = {k: v for k, v in c.items() if v > 1}
    print(f"\n*** {len(multi)} patients have >1 study in this sample ***")
    print("*** GROUP CV FOLDS BY PatientID, NOT StudyInstanceUID ***")
else:
    print("1:1 in this sample -- re-check on the full set before trusting it.")